# Extracting 2023 and 2024 Attributions

This notebook provides functions for fetching team attributions from the *attribution-form* API for 2023 and 2024 (as that API did not exist in the earlier years). The *attribution-form* API provides the UUIDs of team members, a list of tasks that each member performed (from a total of 17 iGEM tasks), descriptions of those tasks (from which we attempt to extract specific task details), and information about external contributors. As team members' attributions were not well structured before the year 2023, extraction of those tasks will be done later on with the use of LLMs.

The *roster* API also provides team members' UUIDs, along with their full names, usernames, roles, etc. Team names, IDs, years, and statuses (e.g., accepted, disqualified) are gathered from the `team_meta_full.tsv` data file, previously extracted from iGEM APIs.

The process begins by extracting attributions for a single team with ID `5033`, including tests to ensure each step functions correctly. Scraping is then performed for all teams across both years.

The returned DataFrame has:

* One row per task (one person can have multiple tasks)
* Columns: `FullName`, `Username`, `TeamID`, `Team`, `Year`, `Role`, `Task`, `TaskDescription`
* Sorted by `FullName`, `TeamID`, and `Year`

**Column descriptions:**

* **FullName** – Full name of the team member
* **Username** – Username of the team member
* **TeamID** – Unique team ID associated with the member
* **Team** – Team name (constant across years, but with unique ID)
* **Year** – Year in which the team and member participated
* **Role** – Member’s role (`Student`, `Student Leader`, `Primary PI`, `Advisor`, `Instructor`, `Secondary PI`, `Other`)
* **Task** – Name of the task performed by the member (one row per member-task pair), chosen from:
  `'Investigation'`, `'Background Research'`, `'Notebook Keeping'`, `'Conceptualization'`, `'Project Administration'`, `'Public Engagement'`, `'Writing'`, `'Safety'`, `'Visualization'`, `'Wiki Coding'`, `'Fundraising'`, `'Analysis'`, `'Data Curation'`, `'Software'`, `'Entrepreneurship'`, `'Hardware'`, `'Other'`
* **TaskDescription** – Text description of the performed task;  can be null due to inconsistencies in attribution form submissions (some tasks may not be marked or described)

Next, data is extracted for external contributors (for future analyses), and then for full task descriptions, since some attribution forms are not well structured and make it difficult to extract individual task descriptions.

The resulting DataFrames are saved in `data/attributions` and `results/attributions`. Exploratory data analysis of team members’ tasks is conducted in `notebooks/2_attributions_EDA.ipynb`.


In [1]:
import requests
import re
import difflib
import pandas as pd

In [2]:
# Reading the team_meta_full.tsv to get all team ids and corresponding team names and years
team_meta_df = pd.read_table(
    "../data/igem_scrapping_2025/team_meta_full.tsv",
    usecols=["TeamID","Team","Year", "Status"]
)

team_meta_df.head()

,TeamID,Team,Status,Year
0,5260,ABOA,accepted,2024
1,5794,ABOA,accepted,2025
2,4831,ABOA-Turku,accepted,2023
3,4068,ABSI_Kenya,accepted,2021
4,2815,ACIBADEM_ISTANBUL,accepted,2018


## 1. Fetching attributions for one team

In [21]:
# Fetches the team’s roster and returns a mapping uuid: (FullName, Role)
# Person's uuid here is the same as teamRosterUUId in application form in the get_team_members_tasks funct
def get_uuid_map(team_id):
    try:
        resp = requests.get(f"https://api.igem.org/v1/teams/{team_id}/roster")
        resp.raise_for_status()
        roster = resp.json()
    except Exception as e:
        raise RuntimeError(f"Failed to fetch roster for team {team_id}: {e}")

    if not isinstance(roster, list):
        raise RuntimeError(f"Malformed roster response for team {team_id}: expected list")

    uuid_map = {}
    for element in roster:
        member = element.get("member") if isinstance(element, dict) else None
        user = member.get("user") if isinstance(member, dict) else None

        # Some roster entries have member.user = null; skip them instead of crashing.
        if not isinstance(user, dict):
            continue

        uuid_map[element["uuid"]] = (
            user.get("publicName"),
            element.get("role"),
            member.get("username")
        )

    return uuid_map

In [22]:
# Testing the return of the previous function for team with id 5333

print(get_uuid_map(5333))

{'4d5afb9b-880a-435c-a016-53510ba945ef': ('Athanasios Saragliadis', 'secondary-pi', 'athanasios'), '77baab6e-f131-43e3-8d83-d598fa0a08e8': ('Bror Johannes Tidemand Ruud', 'student', 'brorjt'), 'd8f2dcdf-b2f3-4986-a49b-f80e9fc60e65': ('Dirk Linke', 'primary-pi', 'dlinke'), '70e68032-e5ac-4914-97ee-5a4133971d91': ('Espen Opseth', 'student', 'espenjop'), 'a001f586-c1fa-4292-b892-053abb942bce': ('Kirsten Borse Haraldsen', 'secondary-pi', 'kbharaldsen'), '9141d63e-cb13-4e5b-9bb8-d7b161881f8f': ('Marianna Khodabandehlou', 'student', 'mariannakhod'), '0dd0f70b-531e-4e4b-a7d7-5792e2832832': ('Marlene Elisabeth Metz', 'student', 'marlenme'), '3e585538-41dd-4b8a-99c8-7fa3977310b1': ('Michelle  Vera Castellanos', 'student', 'michelvc'), 'dbc90597-701e-494c-a829-16aa41589b38': ('Sigurd Graarud', 'student', 'sigurgra'), '6a7086b8-e7bd-43e6-a8bd-9fb95b53531e': ('Sigve', 'student', 'pyrococcus')}


In [23]:
# # Fetches the attribution form for the team.
# Returns a list of dicts, each containing:
#  "teamRosterUUID" (to match against the roster later),
#  "tasks" (the list of task names for each member),
#  "specificTasks" (description of all tasks - split later per task)
# Only includes team members and not external contributors

def get_team_members_tasks(team_id):
    try:
        resp = requests.get(
            f"https://api.igem.org/v1/teams/{team_id}/team-submissions/attribution-form"
        )
        resp.raise_for_status()
        payload = resp.json()
    except Exception as e:
        raise RuntimeError(f"Failed to fetch attribution form for team {team_id}: {e}")

    if not isinstance(payload, list) or not payload:
        raise RuntimeError(f"Malformed attribution form response for team {team_id}: expected non-empty list")

    value = payload[0].get("value") if isinstance(payload[0], dict) else None
    if not isinstance(value, dict):
        raise RuntimeError(f"Malformed attribution form response for team {team_id}: missing value object")

    team_members_tasks = value.get("teamMembers", [])
    return team_members_tasks

In [24]:
# Testing the return of the previous function for team with id 5033

print(get_team_members_tasks(5033))

[{'tasks': ['conceptualization', 'background-research', 'project-administration', 'public-engagement', 'entrepreneurship', 'writing', 'visualization', 'analysis', 'investigation', 'data-curation', 'notebook-keeping'], 'specificTasks': "As a member of the dry lab team, Hannah did (1) - (7) on a weekly/daily basis. Specifically, Hannah worked on the pharmacokinetic simulation of our compound. (2) Conceptualisation - As team lead, Hannah contributed substantially to the complete organisation of the team. In this role she organised the workshops to develop the final idea for the project. The idea for the project originated from Hannah's pitch. (3) Background research - During the development of the idea, a substantial amount of background research was needed to back the idea. She contributed to the development of our initial lab experiments and the well-founded backstory of the project. (7) Project administration - As team lead, Hannah was responsible for organising the management subgroup

In [25]:
# Parses each teamMember’s specificTasks text into individual descriptions for each task
# Returns a df with attributions of each team member (one task per row)
def parse_attributions(team_id, uuid_map, team_members_tasks):
    attributions = []
    for member in team_members_tasks:
        tr_uuid = member["teamRosterUUID"]
        if tr_uuid not in uuid_map:
            continue
        # Match uuids from attribution form to the ones in roster to get task names and descriptions
        full_name, role, username = uuid_map[tr_uuid]
        tasks = member.get("tasks", [])
        description_of_all_tasks  = member.get("specificTasks", "")

        # If there's no specificTasks, all task descriptions for that member should be empty
        if description_of_all_tasks is None:
            for task in tasks:
                attributions.append({
                    "FullName":        full_name,
                    "Username":        username,
                    "RosterUUID":           tr_uuid, 
                    "TeamID":          team_id,
                    "Role":            role,
                    "Task":            task,
                    "TaskDescription": None
                })
            continue

        if len(tasks) == 1:
            attributions.append({
                "FullName":        full_name,
                "Username":        username,
                "RosterUUID":           tr_uuid,
                "TeamID":          team_id,
                "Role":            role,
                "Task":            tasks[0],
                "TaskDescription": description_of_all_tasks.strip()
            })
            continue

        # Split into segments on any "(n)" marker (if there are none, we just get one segment)
        segments = re.split(r"\(\s*\d+\s*\)\s*", description_of_all_tasks)
        '''
        e.g. For specificTasks like this:
        "(1) Conceptualization - Prof. Schwaneberg provided important feedback in designing and administering our project. 
        (2) Fundraising - Prof. Schwaneberg connected us to many potential supporters. 
        (2) Safety - He also provided us with a space in his laboratory and made sure that all the safety requirements were met."
        
        the segments would be:
        [ 
        "",
        "Conceptualization - Prof. Schwaneberg provided important feedback in designing and administering our project. ",
        "Fundraising - Prof. Schwaneberg connected us to many potential supporters. ",
        "Safety - He also provided us with a space in his laboratory and made sure that all the safety requirements were met."
        ]
        '''

        # Build description map
        '''
        Look for matches of tasks and specificTasks like this:

        - only by name (because some teams do not include numbers (1) or they are not numbered correctly
        - include a fuzzy match (to count in typos)
        - for description, keep text after the match of the text name (so after : or -)
        - if a match is not found, leave the description blank (a lot of tasks do not have descriptions)
        - remove double quotes from all descriptions

        '''
        desc_map = {}
        for seg in segments:
            seg = seg.strip()
            if not seg: #skip empty segments (usually the first segment)
                continue
            # Find the first "-" or ":" delimiter
            match_sep = re.match(r"^(?P<label>[^:-]+)[\s]*[-:]\s*(?P<desc>.*)$", seg)
            if not match_sep:
                continue
            label = match_sep.group("label").strip().lower()

            # strip and remove all double quotes from the description (otherwise some desc start with " and some don't)
            desc = match_sep.group("desc").strip().replace('"', '')

            desc_map[label] = desc

        # For each task, lookup its description (blank if missing)
        for task in tasks:
            key = task.replace("-", " ").lower()
            if key in desc_map:
                task_desc = desc_map[key]
            else:
                # Find the closest key in desc_map (counts in typos like Conceptualization vs Conceptualisation)
                candidates = difflib.get_close_matches(key, desc_map.keys(), n=1, cutoff=0.8)
                task_desc = desc_map[candidates[0]] if candidates else ""
            
            attributions.append({
                "FullName":        full_name,
                "Username":        username,
                "RosterUUID":           tr_uuid,
                "TeamID":          team_id,
                "Role":            role,
                "Task":            task,
                "TaskDescription": task_desc
            })

    # Return attributions df with columns "FullName", "TeamID", "Role", "Task", "TaskDescription"
    attributions_df = pd.DataFrame(attributions, columns=[
        "FullName", "Username", "RosterUUID", "TeamID", "Role", "Task", "TaskDescription"
    ])
    return attributions_df

In [26]:
# Wraps previous functions together
def fetch_team_tasks_df(team_id):
    uuid_map = get_uuid_map(team_id)
    team_members_tasks = get_team_members_tasks(team_id)
    return parse_attributions(team_id, uuid_map, team_members_tasks)

In [27]:
# Fetch attributions for team 5033 to check if everything is okay and matches with the https://teams.igem.org/wiki/5033/attributions webpage
attributions_5033_df = fetch_team_tasks_df(5033)

# Merge returned df with the team metadata (merge on TeamId to get the Team and Year values for the team)
desc = attributions_5033_df["TaskDescription"]
# Get non null descriptions so we could clean out the \t separators on those descs
not_na = desc.notna()

attributions_5033_df.loc[not_na, "TaskDescription"] = (
    desc[not_na]
      .str.replace("\t", " ",  regex=False)  # remove tabs \t bc that will be our tsv separator
      .str.replace("\n", " ",  regex=False) # replace new lines with blank spaces
      .str.strip()
)

# print(attributions_5033_df.head())

In [28]:
# this is an Example iGEM team - exclude them 
# fetch_team_tasks_df(5539)

## 2. Comparing returned team 5033 attributions csv file with its attributions webpage

In [50]:
# Manually checking if the names of team members are the same as on the webpage
unique_names = attributions_5033_df["FullName"].unique()
print(unique_names, len(unique_names))

['Hannah Dorn' 'Carl Kallweit' 'Johanna Ewald' 'Jonas M. Westphal'
 'Jonas Lindemann ' 'Lena Buth' 'Bjarne Nies' 'Cherubina'
 'Vanessa Veccari' 'Jackie' 'Emeline' 'Peer Vossen' 'Jan Steinstraßen'
 'Calvino Linkert Herrera' 'Maik Biermann ' 'Saskia Stoye' 'Kilian Mandon'
 'Lisa Brandstäter' 'Carlotta Wolfschmidt' 'Fynn Wadehn'
 'Sebastian Brosch' 'Paul R. F. Cordero' 'Anne Neuß'
 'Jamshid Amiri Moghaddam ' 'Marisa Sárria P. Passos' 'Lars M. Blank'
 'Wolfgang Wiechert' 'Prof. Dr. Lars Lauterbach' 'Ulrich Schwaneberg'] 29


In [51]:
# Fetch the roster, extract all members names, and assert that each shows up in attributions_5033_df 
# if it fails, assert will raise an error
roster = requests.get(f"https://api.igem.org/v1/teams/5033/roster").json()
names_roster = {
    element["member"]["user"].get("publicName") for element in roster
}

names_attr = set(attributions_5033_df["FullName"].dropna())

missing = names_roster - names_attr
assert not missing, f"These roster names never got any rows in attributions_5033_df: {missing}"

In [52]:
# Checking for mismatches of team member name and their role (between the API request response and attributions_5033_df)
role_map = { element["member"]["user"].get("publicName"): element["role"] 
             for element in roster }

mismatches = []
for name, grp in attributions_5033_df.groupby("FullName"):
    if role_map.get(name) != grp["Role"].iloc[0]:
        mismatches.append((name, role_map.get(name), grp["Role"].iloc[0]))

assert not mismatches, f"Role mismatches: {mismatches}"

In [53]:
# Checking if row count of attributions_df matches sum of tasks per team member 
team_members_tasks_5033 = get_team_members_tasks(5033)  
uuid_map_5033 = get_uuid_map(5033)
expected_rows = sum(len(member["tasks"]) for member in team_members_tasks_5033 if member["teamRosterUUID"] in uuid_map_5033)
assert len(attributions_5033_df) == expected_rows, \
    f"Got {len(attributions_5033_df)} rows, but expected {expected_rows}"

After checking that the csv data matches the team attributions webpage, I moved on to expanding the code to all teams for years 2023 and 2024. Some task descriptions are null because teams did not include description for each task, but the matching logic is okay.

## 3. Fetching attributions for all teams in multiple years

In [29]:
team_meta_df["Status"].unique()

array(['accepted', 'withdrawn', 'disqualified'], dtype=object)

In [41]:
# Check only for years where attributions webpages exist (or APIs for them) 
# skip teams whose status is pending, withdrawn or disqualified (include only "Status"="accepted" from team_meta_df)

years_of_interest = [2023, 2024]
# years_of_interest = [2025]

team_ids = (
    team_meta_df.loc[
        (team_meta_df["Year"].isin(years_of_interest)) & 
        (team_meta_df["Status"] == "accepted") &
        (~team_meta_df["Team"].isin(["Example", "example"])),
        "TeamID"
    ]
    .dropna()
    .sort_values()
    .unique()
)

# Fetch tasks for each team 
all_dfs = []
skipped_tids = []

for tid in team_ids:
    try:
        df = fetch_team_tasks_df(tid)
    except Exception as e:
        print(f"Skipping team {tid}: {e}")
        skipped_tids.append(tid)
        continue

    # If the fetched attributions for a team are empty, don't add it (for teams that have an attribution form but it is empty)
    if df.empty:
        print(f"Skipping team {tid}: no attribution data")
        skipped_tids.append(tid)
        continue

    all_dfs.append(df)
    
# Concatenate into one DataFrame (because the fetch_team_tasks_df function returns one df)
if all_dfs:
    attributions_all_years_df = pd.concat(all_dfs, ignore_index=True)
else:
    attributions_all_years_df = pd.DataFrame(
        columns=["FullName", "Username", "RosterUUID", "TeamID", "Role", "Task", "TaskDescription"]
    )

# Merge on TeamID so we get the Year and Team as well
attributions_all_years_df = (
    attributions_all_years_df
    .merge(team_meta_df, on="TeamID", how="left")
    [["FullName", "Username", "RosterUUID", "TeamID", "Team", "Year", "Role", "Task", "TaskDescription"]]
)

# Clean up TaskDescription (only replace \t or \n in non-null descriptions as before)
desc = attributions_all_years_df["TaskDescription"]
not_na = desc.notna()

attributions_all_years_df.loc[not_na, "TaskDescription"] = (
    desc[not_na]
      .str.replace("\t", " ", regex=False)    # remove tabs
      .str.replace("\n", " ", regex=False)    # remove newlines
      .str.strip()                            # trim whitespace
)

# Sort deterministically so row order is stable across runs when underlying API data is unchanged
attributions_all_years_df = attributions_all_years_df.sort_values(
    by=["Year", "TeamID", "Username", "FullName", "Role", "Task", "TaskDescription"],
    kind="stable"
).reset_index(drop=True)

Skipping team 4573: Failed to fetch attribution form for team 4573: 404 Client Error: Not Found for url: https://api.igem.org/v1/teams/4573/team-submissions/attribution-form
Skipping team 4627: Failed to fetch attribution form for team 4627: 404 Client Error: Not Found for url: https://api.igem.org/v1/teams/4627/team-submissions/attribution-form
Skipping team 4635: Failed to fetch attribution form for team 4635: 404 Client Error: Not Found for url: https://api.igem.org/v1/teams/4635/team-submissions/attribution-form
Skipping team 4689: Failed to fetch attribution form for team 4689: 404 Client Error: Not Found for url: https://api.igem.org/v1/teams/4689/team-submissions/attribution-form
Skipping team 4692: Failed to fetch attribution form for team 4692: 404 Client Error: Not Found for url: https://api.igem.org/v1/teams/4692/team-submissions/attribution-form
Skipping team 4709: Failed to fetch attribution form for team 4709: 404 Client Error: Not Found for url: https://api.igem.org/v1/t

In [37]:
skipped_tids

[np.int64(4573),
 np.int64(4627),
 np.int64(4635),
 np.int64(4689),
 np.int64(4692),
 np.int64(4709),
 np.int64(4725),
 np.int64(4741),
 np.int64(4743),
 np.int64(4778),
 np.int64(4784),
 np.int64(4787),
 np.int64(4789),
 np.int64(4866),
 np.int64(4940),
 np.int64(4944),
 np.int64(4948),
 np.int64(5006),
 np.int64(5022),
 np.int64(5023)]

In [ ]:
# Save to results with cleaned task and role columns, and dropped example team

# Capitalize the first letter of each Task entry
attributions_all_years_df['Task'] = (
    attributions_all_years_df['Task']
    .str.replace('-', ' ')
    .str.title()
    .str.strip()
)

# Replace dashes with spaces in Role, then capitalize first letter
attributions_all_years_df['Role'] = (
    attributions_all_years_df['Role']
    .str.replace('-', ' ')
    .str.title()
    .str.strip()
)

attributions_all_years_df['Role'] = (
    attributions_all_years_df['Role']
    .replace({
        'Primary Pi': 'Primary PI',
        'Secondary Pi': 'Secondary PI'
    })
)

# Remove the example team from the resulting df
attributions_all_years_df = attributions_all_years_df[attributions_all_years_df['Team'] != 'Example']

# Deduplicate by member-task within team.
# Keep the row with the longest non-empty TaskDescription when duplicates exist.
dedup_key = ['TeamID', 'RosterUUID', 'Task']
rows_before = len(attributions_all_years_df)

attributions_all_years_df = (
    attributions_all_years_df
    .assign(_desc_len=attributions_all_years_df['TaskDescription'].fillna('').str.len())
    .sort_values(by=['TeamID', 'RosterUUID', 'Task', '_desc_len'], ascending=[True, True, True, False], kind='stable')
    .drop_duplicates(subset=dedup_key, keep='first')
    .drop(columns=['_desc_len'])
    .sort_values(by=['Year', 'TeamID', 'Username', 'FullName', 'Role', 'Task', 'TaskDescription'], kind='stable')
    .reset_index(drop=True)
)

rows_after = len(attributions_all_years_df)
print(f"Removed {rows_before - rows_after} duplicate rows based on {dedup_key}.")

# Save as TSV to results directory
attributions_all_years_df.to_csv("../results/attributions/attributions_2023_2024.tsv", sep="\t", index=False)

Removed 0 duplicate rows based on ['TeamID', 'RosterUUID', 'Task'].
Saved attributions to ../results/attributions/attributions_2023_2024_may_2026.tsv


In [49]:
len(attributions_all_years_df)

85164

In [53]:
attr_check = pd.read_table("../data/attributions/2023_2024_attributions/attributions_2023_2024_january_26_run.tsv", sep="\t")
len(attr_check)

87254

In [51]:
# Check for duplicate member-task pairs (RosterUUID + Task)
dup_mask = attributions_all_years_df.duplicated(subset=["RosterUUID", "Task"], keep=False)
dup_rows = (
    attributions_all_years_df.loc[dup_mask]
    .sort_values(["RosterUUID", "Task", "TeamID", "FullName"])
    .reset_index(drop=True)
)

num_dup_rows = len(dup_rows)
num_dup_pairs = dup_rows[["RosterUUID", "Task"]].drop_duplicates().shape[0]

print(f"Duplicate rows for (RosterUUID, Task): {num_dup_rows}")
print(f"Duplicate (RosterUUID, Task) pairs: {num_dup_pairs}")

if num_dup_rows == 0:
    print("No duplicate member-task pairs found.")
else:
    display(dup_rows)

Duplicate rows for (RosterUUID, Task): 0
Duplicate (RosterUUID, Task) pairs: 0
No duplicate member-task pairs found.


In [52]:
# Check for duplicate member-task pairs (RosterUUID + Task)
dup_mask = attr_check.duplicated(subset=["RosterUUID", "Task"], keep=False)
dup_rows = (
    attr_check.loc[dup_mask]
    .sort_values(["RosterUUID", "Task", "TeamID", "FullName"])
    .reset_index(drop=True)
)

num_dup_rows = len(dup_rows)
num_dup_pairs = dup_rows[["RosterUUID", "Task"]].drop_duplicates().shape[0]

print(f"Duplicate rows for (RosterUUID, Task): {num_dup_rows}")
print(f"Duplicate (RosterUUID, Task) pairs: {num_dup_pairs}")

if num_dup_rows == 0:
    print("No duplicate member-task pairs found.")
else:
    display(dup_rows)

Duplicate rows for (RosterUUID, Task): 4396
Duplicate (RosterUUID, Task) pairs: 2164


,FullName,Username,RosterUUID,TeamID,Team,Year,Role,Task,TaskDescription
0,Ng Tsz Chun,kngtszchun,000b42b2-aa40-4831-9f6d-aad883e9262c,5062,HKU-HongKong,2024,Instructor,Analysis,NaN
1,Ng Tsz Chun,kngtszchun,000b42b2-aa40-4831-9f6d-aad883e9262c,5062,HKU-HongKong,2024,Instructor,Analysis,NaN
2,Ng Tsz Chun,kngtszchun,000b42b2-aa40-4831-9f6d-aad883e9262c,5062,HKU-HongKong,2024,Instructor,Background Research,NaN
3,Ng Tsz Chun,kngtszchun,000b42b2-aa40-4831-9f6d-aad883e9262c,5062,HKU-HongKong,2024,Instructor,Background Research,NaN
4,Ng Tsz Chun,kngtszchun,000b42b2-aa40-4831-9f6d-aad883e9262c,5062,HKU-HongKong,2024,Instructor,Conceptualization,NaN
...,...,...,...,...,...,...,...,...,...
4391,Liu Yue,ly0901,ffb4df54-3a8d-4191-8ec3-242ae6e4b12e,4961,YangzhouNOFLS,2023,Student,Notebook Keeping,NaN
4392,Liu Yue,ly0901,ffb4df54-3a8d-4191-8ec3-242ae6e4b12e,4961,YangzhouNOFLS,2023,Student,Other,NaN
4393,Liu Yue,ly0901,ffb4df54-3a8d-4191-8ec3-242ae6e4b12e,4961,YangzhouNOFLS,2023,Student,Other,NaN
4394,Liu Yue,ly0901,ffb4df54-3a8d-4191-8ec3-242ae6e4b12e,4961,YangzhouNOFLS,2023,Student,Writing,NaN


Issue: API fetched dataset had around 2000 rows more in January 2026, than when refetched in May 2026. The reason is that in the previous code, it duplicated some teams.

In the old loop, when a team fetch failed (attribution form not found), execution did not always continue immediately. The previous team's dataframe could be appended again, inflating counts of rows. This lead to different number of rows each time the API was requested.

In [35]:
# Get unique TeamIDs from both dataframes
unique_teams_attributions = attributions_all_years_df["TeamID"].unique()
print(f"Number of unique TeamID values in attributions_all_years_df: {unique_teams_attributions.shape[0]}")

team_meta_df_2025_accepted = team_meta_df[
    (team_meta_df["Year"] == 2025) & # change year here for other years
    (team_meta_df["Status"] == "accepted") &
    (~team_meta_df["Team"].isin(["Example", "example"]))
]
unique_teams_in_meta = team_meta_df_2025_accepted["TeamID"].unique()
print(f"Number of unique TeamID values in team_meta_df: {unique_teams_in_meta.shape[0]}")

# Find IDs in meta but not in attributions
ids_not_in_attributions = set(unique_teams_in_meta) - set(unique_teams_attributions)

print("TeamIDs not in attributions:")
print(ids_not_in_attributions)

Number of unique TeamID values in attributions_all_years_df: 761
Number of unique TeamID values in team_meta_df: 412
TeamIDs not in attributions:
{np.int64(5540), np.int64(5541), np.int64(5542), np.int64(5543), np.int64(5544), np.int64(5545), np.int64(5546), np.int64(5547), np.int64(5548), np.int64(5549), np.int64(5550), np.int64(5551), np.int64(5553), np.int64(5554), np.int64(5556), np.int64(5560), np.int64(5561), np.int64(5562), np.int64(5563), np.int64(5564), np.int64(5566), np.int64(5567), np.int64(5568), np.int64(5569), np.int64(5570), np.int64(5571), np.int64(5572), np.int64(5573), np.int64(5575), np.int64(5576), np.int64(5577), np.int64(5578), np.int64(5579), np.int64(5580), np.int64(5582), np.int64(5583), np.int64(5584), np.int64(5586), np.int64(5587), np.int64(5588), np.int64(5589), np.int64(5590), np.int64(5591), np.int64(5593), np.int64(5594), np.int64(5595), np.int64(5598), np.int64(5600), np.int64(5601), np.int64(5602), np.int64(5603), np.int64(5604), np.int64(5605), np.in

In [36]:
print(team_meta_df[team_meta_df["TeamID"].isin(skipped_tids) & (team_meta_df["Status"]=="accepted")])

      TeamID             Team    Status  Year
18      4778       AFMU-China  accepted  2023
707     4944          Calgary  accepted  2023
1095    4573       Example-HS  accepted  2023
1440    4725            HKUST  accepted  2023
1684    4692       ICT-Mumbai  accepted  2023
2438    5022   NANJING-NFLSFC  accepted  2023
2513    4635        NEU-CHINA  accepted  2023
2776    4709           Nantes  accepted  2023
2895    4743      PTSH-Taiwan  accepted  2023
3056    4784             Qdai  accepted  2023
3091    4689      REC-CHENNAI  accepted  2023
3114    4741         RUBochum  accepted  2023
3457    4948      Seoul-Korea  accepted  2023
3717    4627     TJUSLS-China  accepted  2023
3956    4787  Tongji-Software  accepted  2023
4316    4866         UIncheon  accepted  2023
4400    5023      UNILA-LatAm  accepted  2023
4761    5006             WIST  accepted  2023
5020    4789       YiYe-China  accepted  2023
5078    4940       iZJU-China  accepted  2023


**For 2023 and 2024**: There are 20 teams whose status is accepted but they are not included in the attributions_2023_2024.csv because they have not yet submitted their attribution form (the webpage just says Note: Team has not submitted their attribution form yet.). Some of them have unstructured attributions on their web page (some in pdfs, some without specific paragraphs), so there is not a uniform way to web scrap attributions to these 20 teams. There is also an example high school team.


**For 2025**: Team 5757 is skipped because it does not have an attributions form. Team 5963 does not have teamMembers in the attribution form, and their attributions website is empty. 

## 4. External Contributions

Making a csv files of external contributions and their tasks within teams (one task per row). Similar functions as before, the application-form API has all the needed info about externals.

In [37]:
def get_external_block(team_id):
    #get external info from API
    url = f"https://api.igem.org/v1/teams/{team_id}/team-submissions/attribution-form"
    resp = requests.get(url)
    resp.raise_for_status()
    payload = resp.json()
    return payload[0]["value"].get("external", [])

In [ ]:
def parse_externals(team_id, externals):
    '''
    Return a df with columns:
      - Name
      - TeamID
      - InstitutionType
      - Relationship
      - Task
      - TaskDescription
      Same task desc logic as before
    '''
    records = []
    for e in externals:
        name = e["name"]
        inst = e.get("institutionType", "")
        rel  = e.get("relationshipToTeam", "")
        tasks = e.get("tasks", [])                 # list of task keys
        spec  = e.get("specificTasks") or ""       # raw text

        if len(tasks) == 1:  
            #If externals have just one task, whatever is written in the specificTask should be added 
            # example: Tasks: Project Administration Specific tasks: Supervised with questions related to laboratory work

            records.append({
                "Name":            name,
                "TeamID":          team_id,
                "InstitutionType": inst,
                "Relationship":    rel,
                "Task":            tasks[0],
                "TaskDescription": spec.strip()  
            })
            continue  

        # 1) split on (n) markers
        segments = re.split(r"\(\s*\d+\s*\)\s*", spec)

        # 2) build a map from normalized task → description
        
        desc_map = {}
        for seg in segments:
            seg = seg.strip()
            if not seg: #skip empty segments (usually the first segment)
                continue
            # Find the first "-" or ":" delimiter
            match_sep = re.match(r"^(?P<label>[^:-]+)[\s]*[-:]\s*(?P<desc>.*)$", seg)
            if not match_sep:
                continue
            label = match_sep.group("label").strip().lower()

            # strip and remove all double quotes from the description (otherwise some desc start with " and some don't)
            desc = match_sep.group("desc").strip().replace('"', '')

            desc_map[label] = desc

        # 3) one output row per task
        for task in tasks:
            key = task.replace("-", " ").lower()
            if key in desc_map:
                task_desc = desc_map[key]
            else:
                # fuzzy match fallback
                cands = difflib.get_close_matches(key, desc_map.keys(), n=1, cutoff=0.8)
                task_desc = desc_map[cands[0]] if cands else ""
            records.append({
                "Name":            name,
                "TeamID":          team_id,
                "InstitutionType": inst,
                "RelationshipToTeam":    rel,
                "Task":            task,
                "TaskDescription": task_desc
            })

    return pd.DataFrame(records, columns=[
        "Name", "TeamID", "InstitutionType", "Relationship", "Task", "TaskDescription"
    ])

In [ ]:
# build the full external_contributions_2023_2024.csv 

#  pick team IDs from team_meta_df same as for attributions

# loop & accumulate
all_ext_dfs = []
skipped = []
for tid in team_ids:
    try:
        externals = get_external_block(tid)
        df_ext = parse_externals(tid, externals)
        all_ext_dfs.append(df_ext)
    except Exception as e:
        print(f"Skipping externals for team {tid}: {e}")
        skipped.append(tid)

# concat
if all_ext_dfs:
    external_all_years_df = pd.concat(all_ext_dfs, ignore_index=True)
else:
    external_all_years_df = pd.DataFrame(
        columns=["Name","TeamID","InstitutionType","Relationship","Task","TaskDescription"]
    )


# merge in Year and Team from team_meta_df
external_all_years_df = (
    external_all_years_df
    .merge(
        team_meta_df[["TeamID","Year","Team"]],
        on="TeamID",
        how="left"
    )
    [["Name","TeamID","Year","Team","InstitutionType","Relationship","Task","TaskDescription"]]
)

# Clean up TaskDescription (only replace \t or \n in non-null descriptions as before)
desc = external_all_years_df["TaskDescription"]
not_na = desc.notna()

external_all_years_df.loc[not_na, "TaskDescription"] = (
    desc[not_na]
      .str.replace("\t", " ", regex=False)    # remove tabs
      .str.replace("\n", " ", regex=False)    # remove newlines
      .str.strip()                            # trim whitespace
)

# Write out as a tsv file (better separator than comma separated values in descriptions)
external_output_path = "../data/attributions/2023_2024_attributions/external_contributions_2023_2024.tsv"
external_all_years_df.to_csv(external_output_path, sep="\t", index=False)

Skipping externals for team 4778: 404 Client Error: Not Found for url: https://api.igem.org/v1/teams/4778/team-submissions/attribution-form
Skipping externals for team 4944: 404 Client Error: Not Found for url: https://api.igem.org/v1/teams/4944/team-submissions/attribution-form
Skipping externals for team 4573: 404 Client Error: Not Found for url: https://api.igem.org/v1/teams/4573/team-submissions/attribution-form
Skipping externals for team 4725: 404 Client Error: Not Found for url: https://api.igem.org/v1/teams/4725/team-submissions/attribution-form
Skipping externals for team 4692: 404 Client Error: Not Found for url: https://api.igem.org/v1/teams/4692/team-submissions/attribution-form
Skipping externals for team 5022: 404 Client Error: Not Found for url: https://api.igem.org/v1/teams/5022/team-submissions/attribution-form
Skipping externals for team 4635: 404 Client Error: Not Found for url: https://api.igem.org/v1/teams/4635/team-submissions/attribution-form
Skipping externals f

In [42]:
unique_external_attributions = external_all_years_df["TeamID"].nunique()
print(f"Number of unique TeamID values in external_all_years_df: {unique_external_attributions}")

Number of unique TeamID values in external_all_years_df: 736


While the attributions_2023_2024.tsv has 765 unique teams, external_contributions_2023_2024.tsv has only 736, due to some teams not having external contributors. Check teams that exist in attributions_all_years_df but not in external_all_years_df.

In [46]:
attrib_teams = set(attributions_all_years_df["TeamID"].unique())
ext_teams    = set(external_all_years_df["TeamID"].unique())

missing_ids = sorted(attrib_teams - ext_teams)

df_missing = pd.DataFrame({"TeamID": missing_ids})

df_missing = df_missing.merge(
    team_meta_df[["TeamID","Team"]],
    on="TeamID",
    how="left"
)

print(df_missing)


    TeamID                  Team
0     4667             BPI-China
1     4670     HongKong-PuiChing
2     4687            HBUT-China
3     4715                  Gifu
4     4732                  Alma
5     4748           Wego-Taipei
6     4770          Barcelona-UB
7     4785         SUSTech-CHINA
8     4788             LZU-CHINA
9     4848                   UNT
10    4896            IITChicago
11    4900    HUST-UEVE-UPSaclay
12    4915           SWUer-China
13    4930          FAU-Erlangen
14    4952              ISPanama
15    4973             PatrasBio
16    5016  MunichBioinformatics
17    5032               Example
18    5193        PuiChing-Macau
19    5203      NYC-Empire-State
20    5211      CHINA-HUBU-WUHAN
21    5249            HBUT-China
22    5305         USTC-Software
23    5368          SCUT-China-B
24    5381             Wentworth
25    5390            SG-Raffles
26    5414                  SEHS
27    5428            USP-Brazil
28    5493           IIT-ROORKEE


I manually checked attribution websites of these team and they indeed do not have any information on external contributors.

## 5. Specific Tasks Full Description

Making attributions_full_specific_tasks_2024_2025 and external_contributions_full_specific_tasks_2024_2025 that for the Task column have a whole list of tasks and for Description they have the entire description from SpecificTasks.

In [ ]:
def build_full_team_attributions_tsv(team_meta_df, years_of_interest, output_path):
    rows = []
    # pick team IDs
    tids = team_meta_df.loc[
        (team_meta_df["Year"].isin(years_of_interest)) &
        (team_meta_df["Status"] == "accepted"),
        ["TeamID", "Year", "Team"]
    ]
    for _, team in tids.iterrows():
        tid, year, team_name = team["TeamID"], team["Year"], team["Team"]
        # roster --> uuid_map
        try:
            roster = requests.get(f"https://api.igem.org/v1/teams/{tid}/roster").json()
        except Exception as e:
            print(f"Skipping team {tid}: Failed to fetch roster: {e}")
            continue
        
        if not roster:
            print(f"Skipping team {tid}: Empty roster")
            continue
            
        uuid_map = {}
        for e in roster:
            if not e.get("member") or not e["member"].get("user"):
                continue  # Skip roster entries with missing member or user data
            uuid_map[e["uuid"]] = (
                e["member"]["user"].get("publicName"),
                e["role"],
                e["member"]["username"]
            )
        
        if not uuid_map:
            print(f"Skipping team {tid}: No valid roster entries")
            continue
            
        # attribution‐form --> members
        try:
            payload = requests.get(
                f"https://api.igem.org/v1/teams/{tid}/team-submissions/attribution-form"
            ).json()
            if not payload or not payload[0] or not payload[0].get("value"):
                print(f"Skipping team {tid}: Malformed attribution form response")
                continue
            members = payload[0]["value"].get("teamMembers", [])
        except Exception as e:
            print(f"Skipping team {tid}: Failed to fetch attribution form: {e}")
            continue

        for m in members:
            tr_uuid = m.get("teamRosterUUID")
            if tr_uuid not in uuid_map:
                continue
            full_name, role, username = uuid_map[tr_uuid]
            tasks = m.get("tasks", [])
            desc  = m.get("specificTasks") or ""
            rows.append({
                "FullName":        full_name,
                "Username":        username,
                "TeamID":          tid,
                "Year":            year,
                "Team":            team_name,
                "Role":            role,
                "Tasks":           tasks,
                "TaskDescription": desc.strip()
            })

    df = pd.DataFrame(rows, columns=[
        "FullName", "Username", "TeamID","Year","Team","Role","Tasks","TaskDescription"
    ])
    
    df["TaskDescription"] = (
        df["TaskDescription"]
          .str.replace('"',  "", regex=False)
          .str.replace("\n", " ", regex=False)
          .str.replace("\t", " ", regex=False)
    )

    # save as TSV
    df.to_csv(output_path, sep="\t", index=False)
    return df

In [ ]:
def build_full_external_contributions_tsv(team_meta_df, years_of_interest, output_path):
    rows = []
    tids = team_meta_df.loc[
        (team_meta_df["Year"].isin(years_of_interest)) &
        (team_meta_df["Status"] == "accepted"),
        ["TeamID", "Year", "Team"]
    ]
    for _, team in tids.iterrows():
        tid, year, team_name = team["TeamID"], team["Year"], team["Team"]
        try:
            payload = requests.get(
                f"https://api.igem.org/v1/teams/{tid}/team-submissions/attribution-form"
            ).json()
            if not payload or not payload[0] or not payload[0].get("value"):
                continue  # Skip teams with malformed responses
            externals = payload[0]["value"].get("external", [])
        except Exception as e:
            print(f"Skipping team {tid}: Failed to fetch external contributions: {e}")
            continue

        for e in externals:
            name = e.get("name", "")
            inst = e.get("institutionType", "")
            rel  = e.get("relationshipToTeam", "")
            tasks = e.get("tasks", [])
            desc  = e.get("specificTasks") or ""
            rows.append({
                "Name":            name,
                "TeamID":          tid,
                "Year":            year,
                "Team":            team_name,
                "InstitutionType": inst,
                "Relationship":    rel,
                "Tasks":           tasks,
                "TaskDescription": desc.strip()
            })

    df = pd.DataFrame(rows, columns=[
        "Name","TeamID","Year","Team",
        "InstitutionType","Relationship","Tasks","TaskDescription"
    ])

    df["TaskDescription"] = (
        df["TaskDescription"]
          .str.replace('"',  "", regex=False)
          .str.replace("\n", " ", regex=False)
          .str.replace("\t", " ", regex=False)
    )

    # save as TSV
    df.to_csv(output_path, sep="\t", index=False)
    return df

In [ ]:
full_team_attributions_df = build_full_team_attributions_tsv(
    team_meta_df,
    years_of_interest,
    "../data/attributions/2023_2024_attributions/attributions_full_specific_tasks_2023_2024.tsv"
)

In [ ]:
full_external_contributions_df = build_full_external_contributions_tsv(
    team_meta_df,
    years_of_interest,
    "../data/attributions/2023_2024_attributions/external_contributions_full_specific_tasks_2023_2024.tsv"
)

In [54]:
unique_full_external_attributions = full_team_attributions_df["TeamID"].nunique()
print(f"Number of unique TeamID values in full_team_attributions_df: {unique_full_external_attributions}")

Number of unique TeamID values in full_team_attributions_df: 765


In [55]:
unique_full_attributions = full_external_contributions_df["TeamID"].nunique()
print(f"Number of unique TeamID values in full_external_contributions_df: {unique_external_attributions}")

Number of unique TeamID values in full_external_contributions_df: 736


## 6. Extracting Attribution Timelines

The attribution-form API also contains info on timelines of each team performing wiki, dry lab, wet lab, recruitment and development categories of tasks. We will create a separate dataframe with each team as a row, and with columns for ending and starting dates of each task category, as well as the duration in weeks. The results will be saved on `data/attributions/attribution_timelines_2023_2024`. 

In [5]:
def get_attributions_timelines(team_id):
    try:
        resp = requests.get(
            f"https://api.igem.org/v1/teams/{team_id}/team-submissions/attribution-form"
        )
        resp.raise_for_status()
        payload = resp.json()
    except Exception as e:
        raise RuntimeError(f"Failed to fetch attribution form for team {team_id}: {e}")
    
    timelines = payload[0]["value"].get("timeline", {})
    return timelines

In [12]:
# Build a df with one row per team, with columns for each timeline segment (start, end, total) for wiki, dryLab, wetLab, recruit, and development

def build_timelines_df(team_id):

    timelines = get_attributions_timelines(team_id) or {}

    def seg(key):
        s = timelines.get(key, {}) if isinstance(timelines, dict) else {}
        return s.get("start"), s.get("end"), s.get("total")

    wiki_s, wiki_e, wiki_t = seg("wiki")
    dry_s,  dry_e,  dry_t  = seg("dryLab")
    wet_s,  wet_e,  wet_t  = seg("wetLab")
    rec_s,  rec_e,  rec_t  = seg("recruit")
    dev_s,  dev_e,  dev_t  = seg("development")

    row = {
        "TeamID": team_id,
        "WikiStartDate": wiki_s,
        "WikiEndDate": wiki_e,
        "WikiDuration": wiki_t,
        "DryLabStartDate": dry_s,
        "DryLabEndDate": dry_e,
        "DryLabDuration": dry_t,
        "WetLabStartDate": wet_s,
        "WetLabEndDate": wet_e,
        "WetLabDuration": wet_t,
        "RecruitmentStartDate": rec_s,
        "RecruitmentEndDate": rec_e,
        "RecruitmentDuration": rec_t,
        "DevelopmentStartDate": dev_s,
        "DevelopmentEndDate": dev_e,
        "DevelopmentDuration": dev_t,
    }

    df = pd.DataFrame([row])

    df = df.merge(team_meta_df[["TeamID", "Team", "Year"]], on="TeamID", how="left")

    df = df.loc[:, [
        "TeamID", "Team", "Year",
        "WikiStartDate", "WikiEndDate", "WikiDuration",
        "DryLabStartDate", "DryLabEndDate", "DryLabDuration",
        "WetLabStartDate", "WetLabEndDate", "WetLabDuration",
        "RecruitmentStartDate", "RecruitmentEndDate", "RecruitmentDuration",
        "DevelopmentStartDate", "DevelopmentEndDate", "DevelopmentDuration",
    ]]
    return df

In [ ]:
# Check only for years where attributions webpages exist (or APIs for them) - 2023 and 2024
# skip teams whose status is pending, withdrawn or disqualified (include only "Status"="accepted" from team_meta_df)

years_of_interest = [2023, 2024]

team_ids = team_meta_df.loc[
    (team_meta_df["Year"].isin(years_of_interest)) &
    (team_meta_df["Status"] == "accepted"),
    "TeamID"
].dropna().unique()

all_dfs = []
skipped_tids = []

for tid in team_ids:
    try:
        df = build_timelines_df(tid)
        all_dfs.append(df)
    except Exception as e:
        print(f"Skipping team {tid}: {e}")
        skipped_tids.append(tid)

# Combine all teams into one DataFrame
if all_dfs:
    final_timelines_df = pd.concat(all_dfs, ignore_index=True)
else:
    final_timelines_df = pd.DataFrame(
        columns=[
            "TeamID", "Team", "Year",
            "WikiStartDate", "WikiEndDate", "WikiDuration",
            "DryLabStartDate", "DryLabEndDate", "DryLabDuration",
            "WetLabStartDate", "WetLabEndDate", "WetLabDuration",
            "RecruitmentStartDate", "RecruitmentEndDate", "RecruitmentDuration",
            "DevelopmentStartDate", "DevelopmentEndDate", "DevelopmentDuration"
        ]
    )

# Convert date columns to datetime
date_cols = [
    "WikiStartDate", "WikiEndDate",
    "DryLabStartDate", "DryLabEndDate",
    "WetLabStartDate", "WetLabEndDate",
    "RecruitmentStartDate", "RecruitmentEndDate",
    "DevelopmentStartDate", "DevelopmentEndDate",
]
final_timelines_df[date_cols] = final_timelines_df[date_cols].apply(
    pd.to_datetime, errors="coerce", utc=True
)

# For each duration column, if missing, compute in rounded weeks between start and end dates
duration_map = {
    "WikiDuration": ("WikiStartDate", "WikiEndDate"),
    "DryLabDuration": ("DryLabStartDate", "DryLabEndDate"),
    "WetLabDuration": ("WetLabStartDate", "WetLabEndDate"),
    "RecruitmentDuration": ("RecruitmentStartDate", "RecruitmentEndDate"),
    "DevelopmentDuration": ("DevelopmentStartDate", "DevelopmentEndDate"),
}

for dur_col, (start_col, end_col) in duration_map.items():
    missing = final_timelines_df[dur_col].isna()
    final_timelines_df.loc[missing, dur_col] = (
        (final_timelines_df.loc[missing, end_col] - final_timelines_df.loc[missing, start_col])
        .dt.total_seconds() / (7 * 24 * 60 * 60)
    )

final_timelines_df[list(duration_map.keys())] = (
    final_timelines_df[list(duration_map.keys())]
    .apply(pd.to_numeric, errors="coerce")
    .round(0)
    .astype("Int64")
)

# Save output
output_path = "../data/attributions/2023_2024_attributions/attribution_timelines_2023_2024.tsv"
final_timelines_df.to_csv(output_path, sep="\t", index=False)

print(f"\n Wrote {len(final_timelines_df)} rows to {output_path}")

Skipping team 4778: Failed to fetch attribution form for team 4778: 404 Client Error: Not Found for url: https://api.igem.org/v1/teams/4778/team-submissions/attribution-form
Skipping team 4944: Failed to fetch attribution form for team 4944: 404 Client Error: Not Found for url: https://api.igem.org/v1/teams/4944/team-submissions/attribution-form
Skipping team 4573: Failed to fetch attribution form for team 4573: 404 Client Error: Not Found for url: https://api.igem.org/v1/teams/4573/team-submissions/attribution-form
Skipping team 4725: Failed to fetch attribution form for team 4725: 404 Client Error: Not Found for url: https://api.igem.org/v1/teams/4725/team-submissions/attribution-form
Skipping team 4692: Failed to fetch attribution form for team 4692: 404 Client Error: Not Found for url: https://api.igem.org/v1/teams/4692/team-submissions/attribution-form
Skipping team 5022: Failed to fetch attribution form for team 5022: 404 Client Error: Not Found for url: https://api.igem.org/v1/t

/var/folders/25/xdw_br_122j5ypnmb925dfjw0000gn/T/ipykernel_34601/2112968645.py:25: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_timelines_df = pd.concat(all_dfs, ignore_index=True)


In [19]:
timelines_df = pd.read_csv(output_path, sep="\t")
timelines_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 765 entries, 0 to 764
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   TeamID                765 non-null    int64  
 1   Team                  765 non-null    object 
 2   Year                  765 non-null    int64  
 3   WikiStartDate         763 non-null    object 
 4   WikiEndDate           763 non-null    object 
 5   WikiDuration          763 non-null    float64
 6   DryLabStartDate       760 non-null    object 
 7   DryLabEndDate         759 non-null    object 
 8   DryLabDuration        759 non-null    float64
 9   WetLabStartDate       761 non-null    object 
 10  WetLabEndDate         757 non-null    object 
 11  WetLabDuration        757 non-null    float64
 12  RecruitmentStartDate  764 non-null    object 
 13  RecruitmentEndDate    764 non-null    object 
 14  RecruitmentDuration   764 non-null    float64
 15  DevelopmentStartDate  7